In [1]:
import polars as pl

# 1. Definimos los mapeos para variables ordinales (orden lógico)
# Esto es crucial para que el modelo entienda jerarquías
map_educacion = {
    "Ninguno": 0,
    "No sabe": 0,
    "No aplica": 0,
    "Primaria incompleta": 1,
    "Primaria completa": 2,
    "Secundaria (Bachillerato) incompleta": 3,
    "Secundaria (Bachillerato) completa": 4,
    "Técnica o tecnológica": 5,
    "Educación profesional incompleta": 6,
    "Educación profesional completa": 7,
    "Postgrado": 8
}

# 2. Lista de columnas que vamos a convertir a binarias (1/0)
cols_binarias = [
    "fami_tieneautomovil", "fami_tienecomputador", "fami_tieneinternet", 
    "fami_tienelavadora", "fami_tieneserviciotv", "cole_bilingue", "cole_sede_principal"
]

def procesar_y_guardar(input_path, output_path):
    # Leer el dataset (usamos separator=";" según tu esquema)
    print("🚀 Cargando datos...")
    df = pl.read_csv(
        input_path, 
        separator=";", 
        null_values=["-", "NAN", "NULL", "", "na"], 
        infer_schema_length=10000
    )

    print("🛠️ Ejecutando ingeniería de características...")
    
    df_final = (
        df
        # --- A. Crear el Periodo y Edad ---
        .with_columns([
            # Combinamos año y semestre en una categoría
            (pl.col("anio").cast(pl.String) + "-" + pl.col("semestre").cast(pl.String)).alias("periodo"),
            
            # EXTRACCIÓN ROBUSTA DEL AÑO:
            # Buscamos 4 dígitos seguidos (\d{4}) en cualquier parte del string
            pl.col("estu_fechanacimiento")
            .str.extract(r"(\d{4})", 0) 
            .cast(pl.Int64)
            .alias("anio_nacimiento")
        ])
        .with_columns(
            # Calculamos la edad usando el año extraído
            estu_edad = pl.col("anio") - pl.col("anio_nacimiento")
        )
        .drop("anio_nacimiento") # Limpiamos la columna temporal
        
        # --- B. Conversión Binaria (Si=1, No=0) ---
        .with_columns([
            pl.when(pl.col(c) == "Si").then(1).otherwise(0).cast(pl.Int8).alias(c)
            for c in cols_binarias
        ])
        
        # --- C. Mapeo Ordinal (Educación) ---
        .with_columns([
            pl.col("fami_educacionmadre").replace(map_educacion, default=None).cast(pl.Int8),
            pl.col("fami_educacionpadre").replace(map_educacion, default=None).cast(pl.Int8)
        ])
        
        # --- D. Limpieza de Estrato (Solo el número) ---
        .with_columns(
            pl.col("fami_estratovivienda")
            .str.extract(r"(\d+)", 1) # Extrae el primer dígito que encuentre
            .cast(pl.Int8)
        )
        
        # --- E. Optimización de Memoria para Categorías ---
        # Convertimos strings repetitivos a Categorical para reducir peso y acelerar entrenamiento
        .with_columns([
            pl.col(c).cast(pl.Categorical)
            for c in [
                "cole_area_ubicacion", "cole_calendario", "cole_caracter", 
                "cole_depto_ubicacion", "cole_genero", "cole_jornada", 
                "cole_naturaleza", "estu_genero", "periodo"
            ]
        ])
        
        # --- F. Selección Final ---
        # Quitamos columnas que ya procesamos o que no aportan (como fecha exacta o ID)
        .drop(["estu_fechanacimiento", "anio", "semestre", "estu_tipodocumento"])
    )

    # 3. Guardar en formato Parquet
    print(f"💾 Guardando en {output_path}...")
    df_final.write_parquet(output_path, compression="zstd") # ZSTD es excelente para datos tabulares
    print("✅ ¡Proceso completado!")

In [2]:
archivo_csv = "/kaggle/input/datasets/lucasiturriago/pruebassaber11/v2_datos_limpios.csv"
archivo_parquet = "saberpro_dataset.parquet"

procesar_y_guardar(archivo_csv, archivo_parquet)

🚀 Cargando datos...
🛠️ Ejecutando ingeniería de características...


/tmp/ipykernel_57/3574366929.py:65: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("fami_educacionmadre").replace(map_educacion, default=None).cast(pl.Int8),
/tmp/ipykernel_57/3574366929.py:66: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("fami_educacionpadre").replace(map_educacion, default=None).cast(pl.Int8)


💾 Guardando en saberpro_dataset.parquet...
✅ ¡Proceso completado!


In [3]:
archivo_csv = "/kaggle/input/datasets/lucasiturriago/pruebassaber11/v2_datos_rural.csv"
archivo_parquet = "saberpro_rural_dataset.parquet"

procesar_y_guardar(archivo_csv, archivo_parquet)

🚀 Cargando datos...
🛠️ Ejecutando ingeniería de características...


/tmp/ipykernel_57/3574366929.py:65: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("fami_educacionmadre").replace(map_educacion, default=None).cast(pl.Int8),
/tmp/ipykernel_57/3574366929.py:66: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("fami_educacionpadre").replace(map_educacion, default=None).cast(pl.Int8)


💾 Guardando en saberpro_rural_dataset.parquet...
✅ ¡Proceso completado!


In [4]:
archivo_csv = "/kaggle/input/datasets/lucasiturriago/pruebassaber11/v2_datos_urbano.csv"
archivo_parquet = "saberpro_urbano_dataset.parquet"

procesar_y_guardar(archivo_csv, archivo_parquet)

🚀 Cargando datos...
🛠️ Ejecutando ingeniería de características...


/tmp/ipykernel_57/3574366929.py:65: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("fami_educacionmadre").replace(map_educacion, default=None).cast(pl.Int8),
/tmp/ipykernel_57/3574366929.py:66: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("fami_educacionpadre").replace(map_educacion, default=None).cast(pl.Int8)


💾 Guardando en saberpro_urbano_dataset.parquet...
✅ ¡Proceso completado!
